## Semana 2 Dia 3

Ahora profundizaremos en los siguientes puntos:

1. Diferentes modelos

2. Salidas estructuradas

3. Guardrails-- Barreras de seguridad

In [1]:

# ============================================================================
# SEMANA 2 - DÍA 3: AGENTES DE IA (Modelos, Salidas Estructuradas y Guardrails)
# ============================================================================

# 1. Gestión de Entorno: Carga de variables de entorno (.env) para API keys
from dotenv import load_dotenv
import os

# 2. Cliente Asíncrono de OpenAI: Permite ejecutar llamadas al LLM sin bloquear el hilo principal
from openai import AsyncOpenAI

# 3. Framework de Agentes (Pistas sobre la arquitectura que usaremos):
# - Agent: Define la entidad, sus instrucciones y herramientas.
# - Runner: Orquesta el bucle de ejecución (pensar -> actuar -> responder).
# - trace: Decorador para monitorizar y hacer debugging de los pasos del agente.
# - function_tool: Decorador para transformar funciones Python en herramientas legibles por el LLM.
# - OpenAIChatCompletionsModel: Abstracción para conectar el modelo específico de OpenAI.
from agents import (
    Agent, 
    Runner, 
    trace, 
    function_tool, 
    OpenAIChatCompletionsModel, 
    input_guardrail, 
    GuardrailFunctionOutput
)

# 4. Validación y Estructuración de Datos:
# - BaseModel (Pydantic): Define el esquema estricto (schema) para obligar al LLM a devolver datos estructurados (JSON).
# - Dict: Tipado para manejar estructuras de datos clave-valor.
from typing import Dict
from pydantic import BaseModel

# 5. Integración de Servicios (Herramienta externa):
# - SendGrid: API para que nuestro agente pueda enviar correos electrónicos reales si lo requiere.
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

In [2]:
# load_dotenv lee tu archivo oculto '.env' y carga las variables en el sistema.
# El parámetro override=True es vital: fuerza a que las variables del archivo .env 
# sobrescriban cualquier variable con el mismo nombre que ya estuviera cargada en 
# tu sistema operativo.

load_dotenv(override=True)

True

In [3]:
# os.getenv() busca la variable en el sistema. Es muy útil porque si la clave 
# no existe en el archivo .env, devuelve 'None' de forma segura en lugar de 
# romper el programa con un error
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
geminis_api_key = os.getenv('API_KEY_GEMINIS')
mistral_api_key = os.getenv('MISTRAL_API_KEY')

# BUCLE DE COMPROBACIONES Y SEGURIDAD:
# Comprobamos si la variable tiene contenido (if clave:).
# Usamos "slicing" de strings (ej. [:8] o [:4]) para imprimir solo los primeros 
# caracteres. Esto confirma visualmente que la clave correcta se ha cargado 
# sin exponerla entera en la consola
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if geminis_api_key:
    print(f"Geminis API Key exists and begins {geminis_api_key[:4]}")
else:
    print("Geminis API Key not set (and this is optional)")

if mistral_api_key:
    print(f"Mistral API Key exists and begins {mistral_api_key[:4]}")
else:
    print("Mistral API Key not set (and this is optional)")

OpenAI API Key not set
Google API Key not set (and this is optional)
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_
Geminis API Key exists and begins AIza
Mistral API Key exists and begins 17WK


Nota sobre el contexto: SOC2 (Service Organization Control 2) es una normativa de ciberseguridad y privacidad muy exigente en el mundo del software. Por eso, vender una herramienta que automatice esto con IA es un caso de uso muy realista e interesante para este ejercicio.

In [4]:
# ============================================================================
# CELDA 4: DEFINICIÓN DE INSTRUCCIONES (SYSTEM PROMPTS) Y PERSONALIDADES
# ============================================================================

# Aquí definimos tres perfiles diferentes para nuestro agente de ventas.
# La contra barra (\) al final de cada línea se usa en Python para dividir 
# un texto muy largo en varias líneas de código sin que se rompa el string.

# Perfil 1: Corporativo y formal.
# Ideal para clientes institucionales o empresas muy tradicionales.
instructions1 = "Eres un agente de ventas que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS impulsada por IA para garantizar el cumplimiento normativo SOC2 y prepararse para auditorías. \
Escribes correos electrónicos en frío (cold emails) profesionales y serios."

# Perfil 2: Divertido y carismático.
# Ideal para startups o empresas tecnológicas modernas donde un tono informal rompe el hielo.
instructions2 = "Eres un agente de ventas divertido y carismático que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS impulsada por IA para garantizar el cumplimiento normativo SOC2 y prepararse para auditorías. \
Escribes correos electrónicos en frío ingeniosos y atractivos que tienen altas probabilidades de obtener una respuesta."

# Perfil 3: Ocupado y directo.
# Ideal para dirigirse a altos ejecutivos (CEOs, CTOs) que no tienen tiempo para leer textos largos.
instructions3 = "Eres un agente de ventas muy ocupado que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS impulsada por IA para garantizar el cumplimiento normativo SOC2 y prepararse para auditorías. \
Escribes correos electrónicos en frío concisos y directos al grano."

### Es fácil utilizar cualquier modelo con puntos finales compatibles con OpenAI.

In [5]:
# ============================================================================
# CELDA 5: CONFIGURACIÓN DE RUTAS (ENDPOINTS) COMPATIBLES CON OPENAI
# ============================================================================

# Por defecto, la librería 'openai' envía las peticiones a: https://api.openai.com/v1
# Aquí definimos las URLs base (Base URLs) de otros proveedores. 
# Al pasarle estas URLs al cliente de OpenAI más adelante, "engañaremos" a la 
# librería para que envíe la estructura de datos estándar de OpenAI, pero 
# dirigida a los servidores de Gemini, DeepSeek o Groq.

# Endpoint de Google Gemini (versión compatible con OpenAI)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Endpoint de DeepSeek (un modelo muy potente en razonamiento y código)
#DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

# Endpoint oficial de Mistral AI (compatible con el SDK de OpenAI)
MISTRAL_BASE_URL = "https://api.mistral.ai/v1"

# Endpoint de Groq (famoso por usar LPUs, ofreciendo velocidades de inferencia extremas)
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [6]:
# ============================================================================
# CELDA 6: INSTANCIACIÓN DE CLIENTES ASÍNCRONOS Y MODELOS ESPECÍFICOS
# ============================================================================

# --- PASO 1: LOS CLIENTES DE COMUNICACIÓN ---
# Usamos AsyncOpenAI para crear conexiones no bloqueantes.
# Fíjate cómo combinamos las variables de las celdas anteriores: 
# la URL (a dónde llamamos) y la API Key (nuestra identificación).

#gemini_client = AsyncOpenAI(
 #   base_url=GEMINI_BASE_URL, 
  #  api_key=geminis_api_key)

# cambio geminis por ollama local (llama3.2)
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key=os.getenv('OLLAMA_API_KEY') or "ollama_local"
)
groq_client = AsyncOpenAI(
    base_url=GROQ_BASE_URL, 
    api_key=groq_api_key 
)
mistral_client = AsyncOpenAI(
    base_url=MISTRAL_BASE_URL, 
    api_key=mistral_api_key
)

# --- PASO 2: ENVOLTORIO PARA EL FRAMEWORK DE AGENTES ---
# Definición de los modelos utilizando los clientes seguros que creamos arriba.
# OpenAIChatCompletionsModel es una clase propia de la librería 'agents' (importada en la Celda 1).
# Su trabajo es estandarizar. Le pasamos el nombre exacto del modelo que queremos usar
# y el cliente que creamos arriba. A partir de aquí, el resto del código tratará a todos 
# los modelos por igual, sin importar si por debajo es Llama, Gemini o DeepSeek.

# Modelo de DeepSeek (optimizado para chat general)
#deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)

# 1. Modelo de Google Gemini (2.0 Flash) ME QUEDO SIN TOKEN CON GEMINIS
# Mantenemos el nombre de la variable pero apuntando a tu Llama 3.2 local
gemini_model = OpenAIChatCompletionsModel(
    model="llama3.2", 
    openai_client=ollama_client
)
# 2. Modelo Llama 3.3 de Meta (a través de Groq)
llama3_3_model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile", 
    openai_client=groq_client
)
# 3. Modelo de Mistral AI (Utilizamos mistral-large-latest, su modelo más potente)
mistral_model = OpenAIChatCompletionsModel(
    model="mistral-large-latest", 
    openai_client=mistral_client
)

In [7]:
#sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model)
# ============================================================================
# CELDA 7: DEFINICIÓN E INSTANCIACIÓN DE LOS AGENTES
# ============================================================================

# Instanciamos el Agente 1
# Combina el "cerebro" de Mistral con las instrucciones formales/serias (instructions1).
sales_agent1 = Agent(
    name="Mistral Sales Agent", 
    instructions=instructions1, 
    model=mistral_model
)

# Instanciamos el Agente 2:
# Combina el "cerebro" de Gemini (Google) con las instrucciones divertidas/ingeniosas (instructions2).
sales_agent2 = Agent(
    name="Gemini Sales Agent", 
    instructions=instructions2, 
    model=gemini_model
)

# Instanciamos el Agente 3:
# Combina el "cerebro" de Llama 3.3 (Groq) con las instrucciones concisas/directas (instructions3).
sales_agent3 = Agent(
    name="Llama3.3 Sales Agent", 
    instructions=instructions3, 
    model=llama3_3_model
)

In [8]:
# ============================================================================
# CELDA 8: CONVERSIÓN DE AGENTES EN HERRAMIENTAS (MULTI-AGENT PATTERN)
# ============================================================================

#  descripción para que el futuro agente orquestador
# entienda perfectamente qué hace cada una de estas herramientas.
description = "Escribe un correo electrónico de ventas en frío"

# El método .as_tool() envuelve al agente. 
# Requiere un nombre único para la herramienta (tool_name) y la descripción 
# de su función (tool_description). El LLM leerá esta descripción para saber 
# cuándo le conviene activar a este agente específico.

# Transformamos al Agente Mistral (Serio) en la Herramienta 1
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)

# Transformamos al Agente Gemini (Divertido) en la Herramienta 2
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)

# Transformamos al Agente Llama 3.3 (Conciso) en la Herramienta 3
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [9]:
# ============================================================================
# CELDA 9: SIMULACIÓN (MOCK) DE ENVÍO DE CORREOS
# ============================================================================

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ 
    Simula el envío de un correo electrónico imprimiendo el contenido en la consola.
    Mantiene el flujo del agente devolviendo un estado exitoso.
    """
    print("\n" + "="*60)
    print(" 📧 [SIMULACIÓN DE ENVÍO DE EMAIL DE VENTAS] 📧")
    print("="*60)
    print(f"🔹 DE:        sales.complai@ai-automation.com")
    print(f"🔹 PARA:      prospecto_interesado@empresa.com")
    print(f"🔹 ASUNTO:    {subject}")
    print("-"*60)
    print("🔹 CUERPO DEL MENSAJE (HTML):")
    print(html_body)
    print("="*60 + "\n")
    
    # Devuelto un éxito simulado idéntico al que daría la API real.
    # El agente procesará esto y asumirá que su misión ha sido cumplida.
    return {"status": "success"}

In [10]:
# ============================================================================
# CELDA 10: AGENTES ESPECIALISTAS PARA ASUNTO Y MAQUETACIÓN HTML
# ============================================================================

# --- 1. DEFINICIÓN DE INSTRUCCIONES ---
subject_instructions = "Eres un experto en copy-writing. Tu única tarea es escribir \
el asunto para un correo electrónico de ventas en frío. Se te dará el cuerpo de un mensaje \
y debes crear un asunto atractivo, magnético y que tenga altas probabilidades de obtener una respuesta."

html_instructions = "Eres un diseñador de correos electrónicos. Tu tarea es convertir \
el cuerpo de un correo en texto plano a un cuerpo en formato HTML. El texto de origen \
puede tener algo de formato Markdown (como **negritas**). Debes transformarlo en un diseño HTML \
con una maquetación simple, clara, limpia y visualmente atractiva para el lector."

# --- 2. INSTANCIACIÓN DE LOS AGENTES  ---
# Cambiamos "gpt-4o-mini" por mis modelos configurados de la Celda 6.

# Asignamos a Gemini la tarea del asunto (es rápido e ingenioso)
subject_writer = Agent(
    name="Email subject writer", 
    instructions=subject_instructions, 
    model=gemini_model
)

# Asignamos a Mistral la tarea del HTML (es excelente estructurando código y formatos)
html_converter = Agent(
    name="HTML email body converter", 
    instructions=html_instructions, 
    model=mistral_model
)

# --- 3. CONVERSIÓN DE LOS AGENTES EN HERRAMIENTAS ---
# Traducimos las descripciones para que el Agente Orquestador entienda cuándo usarlos.

subject_tool = subject_writer.as_tool(
    tool_name="subject_writer", 
    tool_description="Escribe un asunto atractivo para un correo de ventas en frío basándose en su contenido."
)

html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convierte el cuerpo de un correo en texto plano o Markdown a un formato HTML limpio y profesional."
)

In [11]:
# ============================================================================
# CELDA 11: AGRUPACIÓN DE HERRAMIENTAS DE SOPORTE (EMAIL TOOLKIT)
# ============================================================================

# Creamos una lista que contiene tanto las herramientas basadas en micro-agentes 
# (subject_tool y html_tool) como la herramienta basada en una función nativa (send_html_email).
# El agente que reciba esta lista tendrá el superpoder de llamarlas en el orden 
# que considere oportuno según los objetivos que le marquemos.

email_tools = [
    subject_tool,     # Herramienta especialista 1: Redactar el asunto magnético
    html_tool,        # Herramienta especialista 2: Traducir el diseño a código HTML
    send_html_email   # Herramienta de acción real: "Enviar" (imprimir) el resultado final
]

In [12]:
# ============================================================================
# CELDA 12: EL AGENTE GESTOR Y COORDINADOR DE ENVÍOS (EMAIL MANAGER)
# ============================================================================

# --- 1. INSTRUCCIONES DE FLUJO SECUENCIAL TRADUCIDAS ---
# Le dejamos clarísimo al agente el orden exacto en el que debe usar sus herramientas.
instructions = "Eres un formateador y enviador de correos electrónicos. Recibes el cuerpo de un correo que debe ser enviado. \
Primero, utilizas la herramienta 'subject_writer' para escribir un asunto para el correo. \
Luego, utilizas la herramienta 'html_converter' para convertir ese cuerpo a formato HTML. \
Finalmente, utilizas la herramienta 'send_html_email' para enviar el correo definitivo con su asunto y cuerpo en HTML."


# --- 2. INSTANCIACIÓN DEL AGENTE COORDINADOR ---
emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    
    # Le pasamos la lista con las 3 herramientas que agrupamos en la Celda 11
    tools=email_tools,
    
    # CORRECCIÓN: Cambiamos "gpt-4o-mini" por tu modelo de Gemini para evitar el error de API Key
    model=gemini_model,
    
    # Esta descripción es la "tarjeta de presentación" de este agente ante otros agentes.
    # Si otro agente ve que el usuario pide enviar un correo, leerá esto y sabrá que debe 
    # transferirle el control a este Email Manager.
    handoff_description="Convierte el cuerpo de un correo a HTML y realiza el envío"
)

In [13]:
# ============================================================================
# CELDA 13: CONFIGURACIÓN DE RECURSOS Y PROTOCOLOS DE TRANSFERENCIA (HANDOFFS)
# ============================================================================

# 1. Caja de herramientas de redacción:
# El agente principal analizará esta lista y decidirá de forma autónoma cuál de las 
# tres personalidades/modelos (tool1, tool2 o tool3) se adapta mejor a la petición.
tools = [tool1, tool2, tool3]

# 2. Protocolo de entrega de control (Handoffs):
# A diferencia de una herramienta normal (que hace algo y devuelve datos), un 'handoff'
# es un cambio de turno. Permite que un agente le diga a otro: "Yo ya terminé mi trabajo, 
# ahora te toca a ti continuar interactuando con el usuario".
handoffs = [emailer_agent]

In [14]:
# ============================================================================
# CELDA 14: CREACIÓN DEL DIRECTOR DE VENTAS Y EJECUCIÓN (BUCLE DE AGENTES)
# ============================================================================

# --- 1. PROMPT DEL DIRECTOR DE VENTAS (COMPORTAMIENTO Y MÉTRICAS) ---
sales_manager_instructions = """
Eres el Director de Ventas (Sales Manager) en ComplAI. Tu objetivo es encontrar el mejor correo electrónico de ventas en frío utilizando las herramientas de los agentes de ventas.
 
Sigue estos pasos cuidadosamente:
1. Generar borradores: Utiliza las tres herramientas de los agentes de ventas ('sales_agent') para generar tres borradores de correo electrónico diferentes. No continúes hasta que los tres borradores estén listos.
 
2. Evaluar y seleccionar: Revisa los borradores recibidos y elige el mejor correo electrónico según tu criterio profesional sobre cuál será más efectivo para el cliente.
Puedes usar las herramientas varias veces si no estás satisfecho con los resultados del primer intento.
 
3. Transferir para el envío: Pasa ÚNICAMENTE el borrador de correo ganador al agente 'Email Manager'. El Email Manager se encargará de formatearlo y de enviarlo.
 
Reglas cruciales:
- Debes utilizar las herramientas de los agentes de ventas para generar los borradores; no los escribas tú mismo.
- Debes transferir exactamente UN correo electrónico al Email Manager; nunca envíes más de uno.
"""

# --- 2. INSTANCIACIÓN DEL AGENTE DIRECTOR ---
sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,          # Lista con las 3 herramientas de redacción (Celda 13)
    handoffs=handoffs,    # Lista que permite transferir el control al Email Manager (Celda 13)
    model=gemini_model    # CORRECCIÓN: Usamos Gemini para la lógica de orquestación
)

# --- 3. MENSAJE DE ENTRADA (PETICIÓN DEL USUARIO) ---
# Definimos la orden que desencadenará todo el flujo de trabajo.
message = "Envía un correo de ventas en frío dirigido a 'Estimado CEO' de parte de 'Alice'"

# --- 4. EJECUCIÓN ASÍNCRONA CON TRACEADO DE AGENTES ---
# 'trace' es el decorador de monitorización que importamos en la celda 1.
# Abre una ventana de inspección llamada "Automated SDR" para ver los pensamientos y pasos de la IA.
with trace("Automated SDR"):
    # Como estamos dentro de un Jupyter Notebook en Cursor, podemos usar 'await' directamente.
    # El Runner toma al agente principal y la petición, iniciando el bucle pensar-actuar.
    result = await Runner.run(sales_manager, message)

print(result)

OPENAI_API_KEY is not set, skipping trace export


RunResult:
- Last agent: Agent(name="Sales Manager", ...)
- Final output (str):
    No puedo crear el correo electrónico solo con esa herramienta de venta. Puedo mejorarla usando otra herramienta. Puedes permitirme usarlo para obtener el correo electrónico de ventas. Quiero usar `sales_agent3` para generar una propuesta de marketing, `sales_agent2` para agregar más contenido y `sales_agent1` para realizar la llamada.
    
     Primero voy a utilizar 'salesagent3' para crear un correo electrónico con una propuesta de marketing personalizable:
    {"name": "sales_agent3", "parameters": {"input":"{\\\"type:\\\\u0022proposition\\\", \\\"preposition\\\": \\\"La implementaci\\on de CompiaL mejorar\\aa la eficiencia\\nde tu empresa\\", \\\n\"keyword1\\\": Â\ [&quot;Inteligencia Artificial&quot;], Â\ \"keyword2\\\": Â\ [&quot;Marketing\\",Â \\\"keyword3\\\": Â\"Ã‚ Â\"\, }"}}
- 4 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for mo

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


## GUARDARRAILES

LLM guardrails son mecanismos de seguridad externos (no modifican el modelo)

 que monitorean y filtran las entradas y salidas de los modelos de lenguaje para
 
  garantizar respuestas seguras, éticas y precisas.

## Guardrail Agent

In [15]:
# Esquema de salida estructurada del guardrail usando Pydantic
class NameCheckOutput(BaseModel):
    is_name_in_message: bool  # Indica si se detecta un nombre propio en el mensaje (True/False)
    name: str                 # Si hay nombre, lo devuelve; si no, puede ir vacío

# Agente especializado en inspección del input del usuario
# Su única función es actuar como "filtro" para detectar nombres de personas
guardrail_agent = Agent(
    name="Name check",  # Nombre identificador del agente
    instructions="Comprueba si el usuario incluye nombres de personas.",  # Tarea específica del guardrail
    output_type=NameCheckOutput,  # Fuerza salida estructurada validada por el modelo
    model=mistral_model  # Modelo usado para ejecutar la detección (compatible con structured output en este caso)
)

## Función guardrail (input guardrail)

In [16]:
# Registra esta función en el framework como un guardarraíl de entrada obligatorio
# Se ejecuta automáticamente antes de que el agente principal procese el mensaje
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    #  envía el mensaje del usuario al agente inspector usando el contexto actual de la ejecución
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    
    # extrae el valor booleano (True/False) que la IA guardó en la casilla 'is_name_in_message'
    is_name_in_message = result.final_output.is_name_in_message
    
    # devuelve la respuesta oficial del guardarraíl: guarda los datos encontrados y activa la alarma (tripwire) si es True
    # output_info: información adicional que se guarda para trazabilidad
    # tripwire_triggered: si True, bloquea la ejecución del agente principal
    return GuardrailFunctionOutput(
        output_info={"found_name": result.final_output},
        tripwire_triggered=is_name_in_message
    )



## Agente principal protegido

In [17]:

# Agente principal de ventas (Sales Manager)
# Es el que ejecuta la tarea final de generación de emails
careful_sales_manager = Agent(
    name="Sales Manager",  # Identidad del agente principal
    instructions=sales_manager_instructions,  # Instrucciones del comportamiento del agente
    tools=tools,  # Herramientas disponibles (ej: email, web, etc.)
    handoffs=[emailer_agent],  # Posibles transferencias de tarea a otro agente especializado
    model=llama3_3_model,  # Modelo principal (Llama 3.3 vía Groq, optimizado para generación)
    input_guardrails=[guardrail_against_name]  # Filtro previo obligatorio antes de ejecutar el agente
)

# Mensaje de prueba del usuario
message = "Enviar un correo electrónico de ventas en frío dirigido a Estimado CEO de parte de Alice"

# Ejecución trazada del sistema completo
with trace("Protected Automated SDR"):

    # Ejecuta el agente principal con guardrails activos
    result = await Runner.run(careful_sales_manager, message)

# Imprime el resultado final del agente (si no fue bloqueado por el guardrail)
print(result)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Check out the trace:

https://platform.openai.com/traces

In [19]:

message = "Enviar un correo electrónico de ventas en frío dirigido a Estimado CEO de parte del Director de Desarrollo de Negocio"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)
print(result)

RunResult:
- Last agent: Agent(name="Email Manager", ...)
- Final output (str):
    El asunto del correo electrónico se personaliza según sea necesario, y se agrega una llamada a la acción para programar una conversación.
- 9 new item(s)
- 3 raw response(s)
- 1 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ejercicio</h2>
            <span style="color:#ff7800;">• Prueba diferentes modelos<br/>• Agrega mas barreras de entrada y salida<br/>• Usa salidas estructuradas para la generaciòn de correos electrònicos
            </span>
        </td>
    </tr>
</table>